# Encontro 4 · Dentro do multi-agente

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/se-ai-agents-engineering/ai-agents-engineering-v2/blob/main/encontro-04/01-dentro-do-multiagente.ipynb)

**Pergunta do dia:** quando dividir em vários agentes melhora, e quando só encarece?

Até aqui, o assistente da Aurora foi **um** agente. No Encontro 3 ele ganhou uma ferramenta de busca e passou a decidir sozinho quando usá-la. Este notebook faz a pergunta seguinte, que quase todo time faz em algum momento: *e se, em vez de um agente que sabe de tudo, tivermos vários, cada um especialista numa área?*

A ideia parece natural, porque é assim que empresas se organizam. Mas ela tem custo: cada agente a mais é mais uma chamada ao modelo, mais uma passagem de informação em que algo pode se perder e mais um lugar onde o erro pode se esconder. Aqui vamos abrir a caixa e medir.

### O que você vai sair sabendo

- Distinguir três arquiteturas: **agente único**, **roteador + especialistas** e **supervisor + especialistas**.
- Ler um **trace**: quem falou com quem, o que cada agente recebeu e quanto cada um custou.
- Reconhecer duas patologias de sistemas multi-agente: a **diluição de contexto** no handoff e o **ping-pong** entre agentes.
- Responder, com números, se a divisão se paga no caso da Aurora.

### Mapa do notebook

| Etapa | A pergunta da etapa |
|---|---|
| 0 · Ambiente | Está tudo pronto? |
| 1 · Linha de base | Quanto acerta e quanto custa o agente da Aula 3? |
| 2 · Um especialista por dentro | O que muda quando um agente só enxerga a sua área? |
| 3 · O roteador por dentro | Como se decide quem atende? |
| 4 · Roteador + especialistas | A divisão acertou mais ou menos? Onde errou? |
| 5 · O handoff | O que se perde quando um agente passa a pergunta adiante? |
| 6 · O supervisor | E quando a pergunta é de duas áreas? |
| 7 · O ping-pong | Como um sistema de agentes entra em loop sem dar erro? |
| 8 · Síntese | Quando a divisão se paga? |

### Peças que você já conhece

Nada aqui é novo, exceto a coordenação entre agentes:

| Peça | De onde vem |
|---|---|
| As regras de resposta: citar a fonte, dizer "não sei", escalar perguntas de elegibilidade | Encontro 1 |
| O contrato das ferramentas e a camada que valida cada chamada | Encontro 2 |
| A busca híbrida, só com fontes oficiais e só a versão vigente de cada política | Encontro 3 |
| Os casos com gabarito | Demonstração do Encontro 3, mais três novos |

Por isso, se o placar mudar de uma arquitetura para outra, foi a coordenação que mudou.

### Vocabulário do dia

| Termo | Significado aqui |
|---|---|
| **Agente único** | Um agente com todas as ferramentas e acesso a tudo. Também chamado de monolito. |
| **Especialista** | Um agente restrito a uma área: o prompt fala só dela, e a busca só enxerga os espaços dela na wiki. |
| **Roteador** | Decide qual especialista atende. Não busca nem responde. |
| **Supervisor** | Coordena: consulta um ou mais especialistas e monta a resposta final. Também não busca. |
| **Handoff** | O momento em que um agente passa o trabalho a outro, e o que vai junto nessa passagem. |
| **Trace** | O registro passo a passo de um atendimento: cada chamada ao modelo, cada ferramenta, cada handoff. |

### Por que não precisa de chave de API

Cada agente deste notebook é um modelo de linguagem tomando decisões: o roteador escolhendo o especialista, o especialista decidindo o que buscar, o supervisor decidindo quem consultar. Antes da aula, o professor rodou tudo com um modelo de verdade e **gravou cada resposta** (*replay*).

Quando você roda uma célula, o código executa normalmente: roteador, busca, camada e placar. Só na hora de perguntar algo ao modelo ele usa a resposta gravada. Você vê decisões reais, tokens reais e o custo real de cada chamada, sem chave, sem custo e sem esperar.

A única limitação: se você mudar um texto que o modelo lê, não existe resposta gravada para ele. O fim do notebook 02 mostra o que fazer nesse caso.

### Como usar

Abra no Colab pelo botão acima e rode as células **de cima para baixo**, com Shift+Enter. Cada etapa diz no topo o que produz, explica a ideia e termina com uma pergunta. As caixas *Para ir além* são opcionais.

**O entregável do dia está no notebook `02-hands-on-multiagente.ipynb`.** Aqui não há nada a entregar.

## Etapa 0 · Ambiente

No Colab, a célula abaixo baixa o repositório da disciplina, o que leva alguns segundos, e confere a gravação. Na sua máquina, rode o notebook a partir da pasta `encontro-04`.

Apareceu "Tudo certo, pode seguir"? Pode seguir. Deu erro? Use **Ambiente de execução → Reiniciar sessão** e rode a célula de novo.

In [1]:
# Etapa 0 · No Colab, baixa o repositório da disciplina. Na sua máquina, não faz nada.
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.exists("/content/ai-agents-engineering"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/se-ai-agents-engineering/ai-agents-engineering-v2",
                        "/content/ai-agents-engineering"], check=True)
    os.chdir("/content/ai-agents-engineering/encontro-04")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-dotenv"], check=True)
sys.path.insert(0, os.getcwd())

import multiagente as ma
ma.preparar("usar")
casos = ma.carregar_casos()
print(f"{len(casos)} casos. Tudo certo, pode seguir.")

Replay 'usar': 333 respostas gravadas com gpt-4o-mini em 2026-10-12 15:36
Há chave configurada: o que não estiver gravado será chamado ao vivo, com custo.
Busca: 90 trechos da wiki da Aurora, espaços confluence-rh, wiki-beneficios, wiki-financeiro, wiki-rh, wiki-ti
19 casos. Tudo certo, pode seguir.


## Etapa 1 · A linha de base: o agente da Aula 3

**Produz:** `unico`, a rodada do agente único nos 19 casos.

### A ideia

Toda comparação precisa de um ponto de partida medido. O nosso é o agente que decide buscar, da demonstração do Encontro 3: **um** agente, com **uma** ferramenta de busca que enxerga a wiki inteira. Ele recebe a pergunta, decide se busca e com que palavras, e encerra chamando `responder` ou `escalar_para_rh`.

```
pergunta ──► agente único ──► busca (a wiki inteira) ──► resposta com fontes
```

### Os casos

São os 16 casos da demonstração do Encontro 3: perguntas normais, exceções, conflito entre versões de uma política, termo exato, perguntas sem resposta, pedidos de elegibilidade e cumprimentos. A eles se somam três casos novos, escritos para testar a divisão em especialistas:

In [2]:
for caso in casos[-3:]:
    print(f"{caso['id']}  {caso['categoria']:<14} {caso['pergunta']}")

R1  duas-areas     Vou viajar a trabalho para o Rio, com pernoite. Qual o meu limite diário de alimentação? E se a VPN não conectar no Wi-Fi do hotel, o que eu faço?
R2  rota-enganosa  Trabalho de casa três dias por semana. A empresa me ajuda a pagar a internet?
R3  rota-enganosa  Fui de carro próprio visitar um cliente. Quanto recebo de reembolso por km rodado?


- **R1** pede duas áreas numa pergunta só: Financeiro (alimentação na viagem) e TI (VPN no hotel).
- **R2** e **R3** têm **rotas enganosas**: palavras que sugerem uma área quando a resposta está em outra.

Agora, a rodada:

In [3]:
unico = ma.rodar(ma.monolito(), casos)
print()
ma.placar(unico)

  N1  ok   agente único                              rota ok     ok
  N2  ok   agente único                              rota ok     ok
  N4  ok   agente único                              rota ok     ok
  N7  ok   agente único                              rota ok     ok
  E1  ok   agente único                              rota ok     ok
  E2  ok   agente único                              rota ok     ok
  C1  ok   agente único                              rota ok     ok
  C2  ok   agente único                              rota ok     ok
  T1  ok   agente único                              rota ok     ok
  T2  ok   agente único                              rota ok     ok
  A1  ok   agente único                              rota ok     ok
  S1  ERR  agente único                                          desfecho 'escalar', esperado 'nao_sei'
  S2  ERR  agente único                                          desfecho 'escalar', esperado 'nao_sei'
  X1  ok   agente único                     

### Como ler a saída

Cada linha é um caso: o id, `ok` ou `ERR`, os agentes que atuaram, se a rota estava certa e, quando errou, o motivo. Um caso só conta como acerto se cumprir três condições:

1. o **desfecho** é o esperado (responder, escalar ou "não sei");
2. a resposta **cita as páginas** que o gabarito exige;
3. quando o caso pede, a resposta traz o **valor certo**: R$ 160, e não R$ 110; 3 dias, e não 2.

O placar resume a rodada numa linha:

| Coluna | O que mede |
|---|---|
| acertos | casos certos, pela régua acima |
| rota certa | se os agentes acionados enxergavam a parte da wiki de que a dúvida dependia. O agente único enxerga tudo, então aqui a coluna vem sempre cheia. Os casos sem resposta (S1 e S2) ficam de fora |
| chamadas/caso | quantas vezes o modelo foi chamado, em média, por pergunta |
| tokens/caso | tokens de entrada e de saída, em média |
| custo total (US$) | a soma dos 19 casos, ao preço do modelo que fez a gravação |
| custo/acerto (US$) | o custo total dividido pelos acertos. **É o número que importa:** quanto custa uma resposta certa |
| seg/caso | o tempo esperando o modelo, medido na gravação |

**Pergunta para a turma.** Vamos dividir este agente em um roteador e três especialistas: RH (com o Financeiro dentro), TI e Benefícios. Cada especialista só enxerga a sua parte da wiki. **Apostem na enquete:** o acerto sobe, cai ou fica igual? E o custo por acerto?

## Etapa 2 · Um especialista por dentro

**Produz:** nada novo para o placar. Mostra o que muda para um agente quando ele fica preso à sua área.

### A ideia

Um especialista é o mesmo loop do agente único, com duas diferenças:

1. **O prompt fala só da área dele**, e diz o que fazer quando a dúvida não é dela: responder "não sei".
2. **A busca dele só enxerga os espaços da área.** Não é uma instrução no prompt pedindo para não olhar o resto: a ferramenta simplesmente não alcança os outros espaços.

A segunda diferença é a que importa. No Encontro 3, o metadado `espaco` no cabeçalho de cada página da wiki era um filtro de busca. Aqui ele vira **controle de acesso por fonte**, o primeiro dos três problemas de arquitetura do caso da Aurora. Um prompt dizendo "não use dados da folha de pagamento" é um pedido. Uma ferramenta que não enxerga a folha é uma parede.

Na equipe de três especialistas, o Financeiro (viagens e reembolso) fica dentro de RH. Há também a **recepção**, que atende cumprimentos e agradecimentos e não busca nada.

In [4]:
for nome, area in ma.equipe(3).items():
    print(f"{nome:<11} vê {', '.join(area['espacos']) or 'nada':<44} {area['descricao'][:60]}…")

rh          vê wiki-rh, confluence-rh, wiki-financeiro      políticas de pessoas: férias, licenças, jornada e banco de h…
ti          vê wiki-ti                                      suporte de TI: VPN e acesso remoto, códigos de erro, senhas …
beneficios  vê wiki-beneficios                              benefícios: plano de saúde, plano odontológico, vale-refeiçã…
geral       vê nada                                         cumprimentos, agradecimentos e perguntas sobre o que o atend…


Este é o prompt de sistema completo do especialista de TI. As regras do Encontro 1 vêm primeiro; depois vêm a área, os espaços que ele enxerga e o que fazer quando a dúvida não é dele:

In [5]:
area = ma.equipe(3)["ti"]
print(ma.SISTEMA_ESPECIALISTA.format(titulo=area["titulo"], descricao=area["descricao"],
                                     espacos=", ".join(area["espacos"])))

Você responde dúvidas internas de colaboradores da Aurora Tecnologia.

Regras:
1. Use somente as informações dos documentos. Não use conhecimento externo.
2. Em "fontes", cite o id de cada documento que sustenta a resposta.
3. Se dois documentos trazem informações conflitantes sobre o que foi perguntado, não escolha um deles: use o desfecho "nao_sei", explique o conflito e cite os dois documentos.
4. Se a pessoa pergunta se ELA tem direito a um benefício, use o desfecho "escalar": a análise de elegibilidade é feita pelo RH.
5. Se os documentos não trazem a informação, use o desfecho "nao_sei" com fontes vazias.

Você é o especialista de TI do atendimento interno da Aurora. Sua área: suporte de TI: VPN e acesso remoto, códigos de erro, senhas e acesso a sistemas, equipamentos e instalação de software.
Sua ferramenta de busca só enxerga estes espaços da wiki: wiki-ti.
Busque antes de responder. Se a dúvida não for da sua área, ou se a busca não trouxer a informação,
use o desfecho "nao_s

Agora, a mesma pergunta sobre férias para dois especialistas, o de RH e o de TI:

In [6]:
pergunta = "Com quanta antecedência preciso pedir minhas férias?"
for nome in ["rh", "ti"]:
    r = ma.especialista(nome)(pergunta, [], [])
    print(f"especialista {nome}: {r.desfecho} · fontes {r.fontes}\n   {r.resposta[:200]}\n")

especialista rh: responder · fontes ['rh-ferias']
   Você deve solicitar suas férias no Portal do Colaborador com pelo menos 30 dias de antecedência, e a solicitação precisa ser aprovada pela liderança imediata.

especialista ti: escalar · fontes []
   Não concluí em 8 passos. Encaminhado ao RH.



### O que observar

O esperado é que o especialista de RH encontre a política de férias e cite a página, e que o de TI busque na wiki de TI, não encontre nada sobre férias e responda "não sei". Se for isso que aparecer, ele não errou nem inventou: fez exatamente o que o contrato dele pede. Se ele tiver respondido outra coisa, olhe as fontes: de onde veio a resposta?

**Pergunta para a turma.** Essa resposta do especialista de TI é uma falha ou é o comportamento desejado? Em que parte da Aurora essa parede é obrigatória? Guardem a resposta: ela volta no Encontro 6, com dados de folha de pagamento.

## Etapa 3 · O roteador por dentro

**Produz:** a decisão de rota para quatro perguntas, tomada de dois jeitos.

### A ideia

Com vários especialistas, alguém precisa decidir quem atende. Esse é o **roteador**. Ele não busca nem responde: lê a pergunta, lê a lista de especialistas e escolhe um. Vamos comparar dois roteadores:

| Roteador | Como decide | A favor | Contra |
|---|---|---|---|
| **Por modelo** (`llm`) | Um modelo lê a pergunta e a descrição de cada especialista | Entende sinônimos e contexto | Custa uma chamada ao modelo por pergunta e erra de jeitos difíceis de prever |
| **Por regra** | Uma lista de palavras-chave por área: `ferias`, `vpn`, `viagem`… | Não gasta nenhum token, é determinística e fácil de auditar | Não entende nada: "ajuda a pagar a internet" não tem nenhuma das palavras certas |

O roteador por regra é o workflow do Encontro 1, agora no papel de roteador.

O roteador por modelo responde chamando uma ferramenta, `encaminhar`, que ele é obrigado a chamar: o código exige isso pela API, e não por uma frase no prompt. A ferramenta tem quatro campos: o **destino**, a **confiança** (alta, média ou baixa), o **motivo** e a **pergunta reescrita** para o especialista, que será usada na Etapa 5.

### O contrato que o roteador lê

**A descrição de cada especialista é o contrato que o roteador lê**, exatamente como a descrição de uma ferramenta no Encontro 2. Se a descrição for ambígua, o roteador erra, e erra de forma previsível. Este é o prompt completo do roteador:

In [7]:
times = ma.equipe(3)
print(ma.SISTEMA_ROTEADOR.format(lista="\n".join(f"- {k}: {v['descricao']}" for k, v in times.items())))

Você é o roteador do atendimento interno da Aurora Tecnologia.
Você não responde a dúvida: escolhe qual especialista vai atendê-la, e só um.

Especialistas:
- rh: políticas de pessoas: férias, licenças, jornada e banco de horas, trabalho remoto, desligamento, avaliação de desempenho, código de conduta e integração de novos colaboradores; também viagens a trabalho, reembolso de despesas, adiantamento de viagem e cartão corporativo
- ti: suporte de TI: VPN e acesso remoto, códigos de erro, senhas e acesso a sistemas, equipamentos e instalação de software
- beneficios: benefícios: plano de saúde, plano odontológico, vale-refeição, auxílio-creche e demais auxílios, seguro de vida e programa de bem-estar (academia e terapia)
- geral: cumprimentos, agradecimentos e perguntas sobre o que o atendimento faz; não consulta políticas

Chame a ferramenta encaminhar.


Agora, quatro perguntas, cada uma escolhida por um motivo:

- **Auxílio home office:** a resposta está na política de trabalho remoto, na wiki de RH. Mas a palavra "auxílio" aparece na descrição de Benefícios.
- **Reembolso por km:** "reembolso" lembra Benefícios (reembolso de consulta), mas a regra está na política de viagens, que é do Financeiro.
- **R1:** duas perguntas em uma, alimentação na viagem e VPN no hotel.
- **Um agradecimento**, que não é de área nenhuma.

In [8]:
perguntas = [
    "Qual o valor do auxílio home office?",
    "Fui de carro próprio visitar um cliente. Quanto recebo de reembolso por km rodado?",
    casos[-3]["pergunta"],                       # R1: duas áreas numa pergunta só
    "Obrigado, era isso mesmo que eu precisava!",
]
for pergunta in perguntas:
    llm = ma.rotear_com_llm(pergunta, times, [])
    regra = ma.rotear_com_regra(pergunta, times, [])
    print(pergunta[:90])
    print(f"   modelo → {llm['destino']:<11} ({llm['confianca']})  {llm['motivo'][:70]}")
    print(f"   regra  → {regra['destino']:<11} ({regra['confianca']})  {regra['motivo'][:70]}\n")

Qual o valor do auxílio home office?
   modelo → beneficios  (alta)  Dúvida específica sobre o valor do auxílio home office.
   regra  → rh          (baixa)  palavras: ['home office', 'auxilio']

Fui de carro próprio visitar um cliente. Quanto recebo de reembolso por km rodado?
   modelo → rh          (alta)  Dúvida sobre reembolso de despesas de viagem.
   regra  → rh          (alta)  palavras: ['reembolso', 'km']

Vou viajar a trabalho para o Rio, com pernoite. Qual o meu limite diário de alimentação? E
   modelo → rh          (alta)  Dúvida sobre limites de viagem a trabalho.
   regra  → rh          (alta)  palavras: ['vpn', 'wi-fi', 'viajar', 'hotel', 'alimentacao']

Obrigado, era isso mesmo que eu precisava!
   modelo → geral       (alta)  O colaborador está agradecendo pelo atendimento.
   regra  → geral       (alta)  palavras: ['obrigad']



### Como ler a saída

Para cada pergunta, uma linha por roteador: o destino escolhido, a confiança e o motivo. No roteador por regra, o motivo é a lista de palavras-chave encontradas. Quando duas áreas empatam em palavras, a regra marca a confiança como baixa; quando não encontra nenhuma, manda para a recepção.

Lembrete: com três especialistas, o Financeiro mora dentro de RH. Então **rh** é a rota certa tanto para o auxílio home office quanto para o reembolso por km.

**Pergunta para a turma.**

- Onde o modelo e a regra discordam? Qual dos dois errou?
- Se o modelo mandou o auxílio home office para Benefícios, que palavra da descrição dos especialistas puxou a pergunta para lá? Como você reescreveria essa descrição?
- O R1 pede duas áreas, e o roteador só pode escolher uma. O que acontece com a parte que ficou de fora?

## Etapa 4 · Roteador + especialistas, nos mesmos casos

**Produz:** `rotas`, a rodada do roteador por modelo com três especialistas.

### A ideia

Agora o desenho completo: o roteador escolhe **um** especialista, e esse especialista atende sozinho, do começo ao fim, só com a sua parte da wiki.

```
                         ┌──► especialista RH          (wiki-rh, confluence-rh, wiki-financeiro)
pergunta ──► roteador ───┼──► especialista TI          (wiki-ti)
                         ├──► especialista Benefícios  (wiki-beneficios)
                         └──► recepção                 (sem busca)
```

### Os dois jeitos de errar

Num sistema com roteador, há dois lugares onde o erro pode nascer, e a coluna **rota certa** separa os dois:

- **Rota errada:** o roteador mandou a pergunta para quem não enxerga a resposta. O especialista pode se comportar perfeitamente, dizendo "não sei", e mesmo assim o caso erra.
- **Rota certa, resposta errada:** o especialista certo recebeu a pergunta e errou, do mesmo jeito que o agente único erraria.

Separar os dois diz onde mexer: na descrição dos especialistas, no primeiro caso, ou no especialista, no segundo.

In [9]:
rotas = ma.rodar(ma.roteador_especialistas("llm", 3), casos)
print()
ma.placar(unico, rotas)

  N1  ok   especialista rh                           rota ok     ok
  N2  ok   especialista rh                           rota ok     ok
  N4  ok   especialista ti                           rota ok     ok
  N7  ok   especialista beneficios                   rota ok     ok
  E1  ok   especialista rh                           rota ok     ok
  E2  ok   especialista rh                           rota ok     ok
  C1  ok   especialista rh                           rota ok     ok
  C2  ERR  especialista beneficios                   ROTA ERRADA desfecho 'escalar', esperado 'responder'
  T1  ok   especialista ti                           rota ok     ok
  T2  ok   especialista beneficios                   rota ok     ok
  A1  ok   especialista beneficios                   rota ok     ok
  S1  ok   especialista beneficios                               ok
  S2  ok   especialista geral                                    ok
  X1  ok   especialista beneficios                   rota ok     ok
  P1  ok  

### Quais casos mudaram?

`onde_mudou` lista só os casos em que uma arquitetura acertou e a outra errou. É ali que a divisão fez diferença, para o bem ou para o mal. Os casos que as duas acertam, ou que as duas erram, não dizem nada sobre a divisão.

In [10]:
ma.onde_mudou(unico, rotas)

caso  agente único (Aula 3)               roteador llm · 3 esp · original · 
C2    ok                                  ERR desfecho 'escalar', esperado '
S1    ERR desfecho 'escalar', esperado '  ok
S2    ERR desfecho 'escalar', esperado '  ok
R1    ok                                  ERR não citou ti-vpn-e-acesso-remo


### Como ler um trace

Para entender um caso, abra o trace dele. Cada linha é um evento, na ordem em que aconteceu:

| Linha do trace | O que significa |
|---|---|
| `[roteador] → rh (confiança alta) · motivo` | A rota que o roteador escolheu e por quê |
| `[roteador] passa adiante → especialista rh: “…”` | O texto **exato** que o especialista recebeu |
| `[especialista rh] acessa: wiki-rh, …` | Os espaços da wiki que esse agente enxerga |
| `[especialista rh] modelo pediu: buscar_politicas_rh (812 tokens de entrada, 41 de saída)` | O que o modelo decidiu fazer naquele passo, e quanto leu e escreveu para isso |
| `[especialista rh] buscar_politicas_rh({"pergunta": "…"}) → ['rh-ferias']` | A consulta que o modelo escreveu e as páginas que a busca devolveu |
| `[especialista rh] encerrou: responder · fontes [...] · “…”` | O desfecho, as fontes citadas e o começo da resposta |
| `custo por agente` | Ao final: chamadas, tokens e custo de cada agente |

O recuo maior marca quem trabalha a mando de outro agente. Troque o id abaixo pelo de um caso que mudou:

In [11]:
ma.mostrar_trace(rotas, "C2")          # troque pelo id de um caso da lista acima

C2 · Qual o valor do auxílio home office?
arquitetura: roteador llm · 3 esp · original · incerto→segue

  [roteador] modelo pediu: encaminhar   (347 tokens de entrada, 45 de saída)
  [roteador] → beneficios (confiança alta)  · Dúvida específica sobre o valor do auxílio home office.
  [roteador] passa adiante → especialista beneficios: “Qual o valor do auxílio home office?”
      [especialista beneficios] acessa: wiki-beneficios
      [especialista beneficios] recebeu: Qual o valor do auxílio home office?
      [especialista beneficios] modelo pediu: buscar_politicas_beneficios   (606 tokens de entrada, 23 de saída)
      [especialista beneficios] buscar_politicas_beneficios({"pergunta": "valor auxílio home office"})  →  ['beneficios-auxilio-creche', 'beneficios-vale-refeicao']
      [especialista beneficios] modelo pediu: buscar_politicas_beneficios   (969 tokens de entrada, 23 de saída)
      [especialista beneficios] buscar_politicas_beneficios({"pergunta": "auxílio home office"})  →

**Pergunta para a turma.**

- Volte à aposta. Quem acertou mais, e por quanto?
- Nos casos que mudaram, o erro foi de rota ou do especialista?
- O custo por acerto subiu ou desceu? Quantas chamadas ao modelo a divisão acrescentou por pergunta? De onde vem essa chamada a mais?

## Etapa 5 · O handoff, aberto

**Produz:** `resumo`, a mesma arquitetura, mas com o especialista recebendo a pergunta reescrita pelo roteador.

### A ideia

**Handoff** é o momento em que um agente passa o trabalho a outro, e em todo handoff alguém decide o que vai junto. Até aqui, o especialista recebeu a pergunta original, palavra por palavra. A alternativa, comum em tutoriais e frameworks, é o roteador reescrever a pergunta "para o especialista entender melhor": mais curta, só com o que importa.

Parece uma boa ideia. O risco é que quem resume decide o que importa, e pode cortar justamente o detalhe que decidia a resposta: a cidade do pernoite (R$ 110 ou R$ 160?), o código do erro, a categoria do plano. Isso se chama **diluição de contexto**, e é silenciosa: nenhum erro aparece, e o especialista responde com toda a confiança à pergunta que recebeu.

É o mesmo problema de um atendimento em níveis, em que cada nível resume o chamado para o próximo.

In [12]:
resumo = ma.rodar(ma.roteador_especialistas("llm", 3, handoff="resumo_do_roteador"), casos, mostrar=False)
ma.placar(rotas, resumo)
print()
ma.onde_mudou(rotas, resumo)

arquitetura                                      acertos  rota certa  chamadas/caso  tokens/caso  custo total (US$)  custo/acerto (US$)  seg/caso
roteador llm · 3 esp · original · incerto→segue  16/19    14/17       3.1            2588         0.0599             0.00375             3.2     
roteador llm · 3 esp · resumo · incerto→segue    16/19    14/17       3.1            2581         0.0598             0.00374             3.1     

caso  roteador llm · 3 esp · original ·   roteador llm · 3 esp · resumo · in
(as duas acertam e erram nos mesmos casos)


Lado a lado, a pergunta original e o que o especialista recebeu, quando o roteador reescreve:

In [13]:
# O que o especialista recebeu em cada caso, quando o roteador reescreve
for linha in resumo["linhas"]:
    passado = next(e["texto"] for e in linha["trace"] if e["evento"] == "handoff")
    marca = "ok " if linha["acertou"] else "ERR"
    print(f"{linha['id']:<4}{marca} {linha['pergunta'][:60]:<62}→ {passado[:60]}")

N1  ok  Quantos dias de licença-paternidade eu tenho?                 → Quantos dias de licença-paternidade um colaborador tem?
N2  ok  Com quanta antecedência preciso pedir minhas férias?          → Com quanta antecedência preciso pedir minhas férias?
N4  ok  Em quanto tempo a TI conserta ou repõe um notebook quebrado?  → Qual é o prazo para conserto ou reposição de um notebook que
N7  ok  Quantas sessões de terapia a empresa paga?                    → Quantas sessões de terapia a empresa paga?
E1  ok  Qual o limite diário de alimentação numa viagem com pernoite  → Qual o limite diário de alimentação numa viagem com pernoite
E2  ok  Até quanto posso gastar por dia com comida numa viagem a tra  → Qual é o limite diário de gastos com comida em viagens a tra
C1  ok  Quantos dias de home office posso fazer por semana?           → Quantos dias de home office posso fazer por semana?
C2  ERR Qual o valor do auxílio home office?                          → Qual é o valor do auxílio home office

**Pergunta para a turma.**

- Procure um caso em que a reescrita tirou um detalhe que decidia a resposta. O caso errou?
- E se nenhum caso mudou de resultado? Olhe a coluna de tokens: o roteador escreveu uma pergunta nova em toda chamada. Isso se pagou?
- Se fosse obrigatório resumir (porque a conversa original é longa demais, por exemplo), que regra vocês dariam ao roteador sobre o que **nunca** pode sair do resumo?

## Etapa 6 · O supervisor: mais de um especialista por pergunta

**Produz:** `sup`, a rodada do supervisor com três especialistas.

### A ideia

O roteador tem uma limitação de projeto: escolhe **um** especialista, e uma pergunta de duas áreas fica pela metade. O **supervisor** resolve isso mudando o papel de quem coordena:

- ele não busca nada;
- cada especialista vira uma **ferramenta** dele: `consultar_rh`, `consultar_ti`, `consultar_beneficios`;
- ele pode consultar quantos quiser, um depois do outro, e monta a resposta final com o que os especialistas trouxeram, citando as fontes que eles citaram.

```
pergunta ──► supervisor ──► consultar_rh ──► especialista RH (busca e responde) ──┐
                 ▲                                                                │
                 └──────────── desfecho + fontes + resposta ◄─────────────────────┘
                 │
                 ├──► consultar_ti ──► especialista TI ──► …
                 │
                 └──► responder (com as fontes de todos)
```

### O custo escondido

A cada passo, o supervisor relê a conversa inteira: a pergunta, o que ele já pediu e tudo o que cada especialista devolveu. Quanto mais especialistas ele consulta, mais longa fica essa conversa, e mais tokens de entrada custa cada passo seguinte. Esse é o **custo de coordenação**, e ele aparece na coluna tokens/caso.

In [14]:
sup = ma.rodar(ma.supervisor(3), casos)
print()
ma.placar(unico, rotas, sup)

  N1  ok   especialista rh                           rota ok     ok
  N2  ok   especialista rh                           rota ok     ok
  N4  ok   especialista ti                           rota ok     ok
  N7  ok   especialista beneficios                   rota ok     ok
  E1  ok   especialista rh                           rota ok     ok
  E2  ok   especialista rh                           rota ok     ok
  C1  ok   especialista rh                           rota ok     ok
  C2  ok   especialista rh                           rota ok     ok
  T1  ok   especialista ti                           rota ok     ok
  T2  ok   especialista beneficios                   rota ok     ok
  A1  ok   especialista beneficios                   rota ok     ok
  S1  ok   especialista beneficios                               ok
  S2  ok   especialista rh > especialista ti > especialista beneficios            ok
  X1  ok   -                                         ROTA ERRADA ok
  P1  ERR  -                   

Agora, o R1 aberto no trace. Repare em três coisas:

1. quantos especialistas o supervisor consultou;
2. o que ele passou para cada um: a pergunta inteira ou só a parte daquela área?
3. no fim, quanto cada agente custou, e quanto custou o próprio supervisor.

In [15]:
ma.mostrar_trace(sup, "R1")

R1 · Vou viajar a trabalho para o Rio, com pernoite. Qual o meu limite diário de alimentação? E se a VPN não conectar no Wi-Fi do hotel, o que eu faço?
arquitetura: supervisor · 3 esp

  [supervisor] recebeu: Vou viajar a trabalho para o Rio, com pernoite. Qual o meu limite diário de alimentação? E se a VPN não conectar no Wi-Fi do hotel, o que eu faço?
  [supervisor] modelo pediu: consultar_rh, consultar_ti   (754 tokens de entrada, 78 de saída)
  [supervisor] passa adiante → especialista rh: “Qual é o limite diário de alimentação para viagens a trabalho com pernoite?”
      [especialista rh] acessa: wiki-rh, confluence-rh, wiki-financeiro
      [especialista rh] recebeu: Qual é o limite diário de alimentação para viagens a trabalho com pernoite?
      [especialista rh] modelo pediu: buscar_politicas_rh   (663 tokens de entrada, 29 de saída)
      [especialista rh] buscar_politicas_rh({"pergunta": "limite diário alimentação viagens a trabalho com pernoite"})  →  ['fin-adiantamento-de-

**Pergunta para a turma.**

- O R1 saiu certo? Quantas chamadas ao modelo ele custou, e quem gastou mais tokens?
- Nos casos de uma área só, o supervisor ajudou em alguma coisa, ou só acrescentou chamadas?
- **Multi-agente é microserviço.** Cada handoff é uma chamada remota, com contrato, latência e falha própria. Em DDD, dividir só se paga quando o limite entre os contextos é real. Aqui, o limite entre RH, TI e Benefícios é real? O que o torna real: o prompt diferente ou a fonte diferente?

## Etapa 7 · Uma patologia, ao vivo: o ping-pong

**Produz:** `pingue`, o supervisor com duas instruções frágeis.

### A ideia

Sistemas com vários agentes falham de jeitos que um agente único não falha. Um dos mais comuns é o **ping-pong**: dois agentes devolvem o trabalho um ao outro, e ninguém encerra. Para provocá-lo, demos duas instruções que, isoladas, parecem razoáveis:

- **aos especialistas:** na dúvida, não responda; peça ao supervisor que confirme antes com a outra área. Parece prudência.
- **ao supervisor:** nunca encerre com "não sei"; insista até alguém responder. Parece compromisso com o colaborador.

Juntas, elas formam um loop: o especialista devolve, o supervisor consulta outro, que também devolve, e o supervisor volta ao primeiro. Repare que **nada quebra**: não há exceção nem mensagem de erro. O sistema só continua gastando.

In [16]:
print(ma.FRAGIL_ESPECIALISTA)
print(ma.FRAGIL_SUPERVISOR)


Se a dúvida tocar, mesmo de leve, em outra área além da sua, não responda nada: use o desfecho
"nao_sei" e peça ao supervisor que confirme antes com o especialista da outra área.

Nunca encerre com "nao_sei". Se um especialista pedir confirmação de outra área, consulte essa outra
área e depois volte ao primeiro especialista com o que ela disse. Repita até alguém responder.


Os dois casos de teste: o R1, que é de duas áreas, e o C2, que parece de duas. O placar compara o supervisor normal com o frágil:

In [17]:
dois = [c for c in casos if c["id"] in ("R1", "C2")]
pingue = ma.rodar(ma.supervisor(3, fragil=True), dois)
print()
ma.placar(ma.rodar(ma.supervisor(3), dois, mostrar=False), pingue)

  C2  ok   especialista rh                           rota ok     ok
  R1  ok   especialista rh > especialista ti         rota ok     ok

arquitetura                  acertos  rota certa  chamadas/caso  tokens/caso  custo total (US$)  custo/acerto (US$)  seg/caso
supervisor · 3 esp           2/2      2/2         5.0            4682         0.0126             0.00629             6.5     
supervisor · 3 esp · frágil  2/2      2/2         5.0            4926         0.0131             0.00657             6.7     


O trace do R1 com o supervisor frágil, passo a passo:

In [ ]:
ma.mostrar_trace(pingue, "R1")

**Pergunta para a turma.** O que parou o ping-pong? Sem isso, quanto custaria esta pergunta? Que métrica de produção avisaria vocês de que isso está acontecendo?

<details>
<summary><b>Resposta (abra depois de discutir)</b></summary>

O que parou o loop foi o **critério de parada**: o limite de passos (`MAX_PASSOS = 8`) que o código impõe ao supervisor. Quando o limite estoura, o atendimento é encerrado e escalado para o RH, com a mensagem "Não concluí em 8 passos". É o componente de agente mais esquecido, aquele do Encontro 1.

Sem ele, o loop só pararia quando a conversa estourasse o tamanho máximo que o modelo aceita, depois de muitas chamadas pagas. As métricas que denunciam o problema em produção são as **chamadas por atendimento** e o **custo por atendimento**: uma cauda longa nessas duas distribuições é o sinal clássico de loop. Medir isso é o tema do Encontro 5.

</details>

## Etapa 8 · Síntese

**Produz:** o placar das quatro arquiteturas, lado a lado.

In [18]:
ma.placar(unico, rotas, resumo, sup)

arquitetura                                      acertos  rota certa  chamadas/caso  tokens/caso  custo total (US$)  custo/acerto (US$)  seg/caso
agente único (Aula 3)                            16/19    17/17       2.6            3007         0.0655             0.00409             2.7     
roteador llm · 3 esp · original · incerto→segue  16/19    14/17       3.1            2588         0.0599             0.00375             3.2     
roteador llm · 3 esp · resumo · incerto→segue    16/19    14/17       3.1            2581         0.0598             0.00374             3.1     
supervisor · 3 esp                               17/19    15/17       3.9            3370         0.079              0.00465             4.2     


### Quando a divisão se paga

Juntando o que o placar e os traces mostraram, a divisão em vários agentes se paga quando há pelo menos uma destas razões, e ela aparece nos números:

1. **O limite é real.** Cada especialista tem fonte, permissão ou ferramenta própria, e não só um prompt diferente. É o critério de DDD: um contexto delimitado com fronteira de verdade.
2. **A parede é exigência.** Há uma regra de acesso por fonte que um agente único não garantiria. No caso da Aurora, este é o argumento mais forte para dividir.
3. **O ganho aparece no placar.** Mais acerto ou menos risco, e não só um código mais organizado, num valor que paga as chamadas a mais.

Quando nada disso vale, o especialista é **acoplamento com passo extra**: o mesmo trabalho, mais caro e com mais lugares para errar. É a Tese 2 da disciplina de novo: antes de acrescentar autonomia, ou agentes, pergunte se ela se paga.

### O que levar daqui

- Toda divisão tem um custo fixo: a coordenação. Ele aparece em chamadas, tokens e tempo, mesmo quando o acerto não muda.
- O roteador erra por causa da **descrição** dos especialistas, que é um contrato como qualquer outro.
- O handoff é um ponto de perda de informação. Passe adiante o original sempre que puder.
- Em sistema multi-agente, as falhas mais caras não dão erro: aparecem só no trace e na fatura.

**Agora, no notebook 02,** o grupo mexe numa coisa de cada vez e decide com o próprio placar.

> **Para ir além (opcional).** Abra `multiagente.py` e procure `AREAS`. As descrições dos especialistas são as que um time escreveria na primeira vez. Que frase você mudaria para o roteador acertar o auxílio home office? Testar a mudança exige chave, porque o modelo vai ler um texto que não está na gravação. O fim do notebook 02 mostra como.